In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🔬 Research Workflow with Persistent Memory

## What We're Building

A multi-session research assistant that builds a topic brief
over several invocations, retaining notes between sessions:

1. **Gather sources** — take user's topic + new context each session
2. **Extract insights** — pull key findings from new material
3. **Update brief** — merge new insights into a running research brief
4. **Identify gaps** — suggest what to explore next

## Architecture

```
Session 1: User provides topic + initial sources
           ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
           │ Gather   │──▶│ Extract  │──▶│ Update   │──▶│ Identify │
           │ Sources  │   │ Insights │   │ Brief    │   │ Gaps     │
           └──────────┘   └──────────┘   └──────────┘   └──────────┘
                                              ▲               │
                                              │    checkpoint  │
                                              └───── saves ◄──┘
Session 2: User provides more sources
           ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
           │ Gather   │──▶│ Extract  │──▶│ Update   │──▶│ Identify │
           │ Sources  │   │ Insights │   │ Brief    │   │ Gaps     │
           └──────────┘   └──────────┘   │ (merges) │   └──────────┘
                                         └──────────┘
```

## Key LangGraph Concept: Checkpointer Persistence

Using `MemorySaver` with the same `thread_id` across multiple `invoke()` calls
lets us resume exactly where we left off. The state from session 1 is available
in session 2 automatically.

## Stack
- **LangGraph** — pipeline with MemorySaver persistence
- **Ollama** — local LLM

## Step 1 — Imports & Setup

In [ ]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.2)
print("All imports successful!")

## Step 2 — State Definition

We use `Annotated[list, operator.add]` for `session_log` — this means
LangGraph **appends** new entries instead of replacing the list. Perfect
for accumulating notes across sessions.

In [ ]:
class ResearchState(TypedDict):
    topic: str                                         # Research topic
    new_context: str                                   # New material provided this session
    latest_insights: str                               # Insights extracted this round
    running_brief: str                                 # Accumulated research brief
    gaps: str                                          # Current knowledge gaps
    session_log: Annotated[list[str], operator.add]    # Append-only log of sessions
    session_number: int                                # Which session we're on


print("State defined with append-only session_log")

## Step 3 — Define Nodes

In [ ]:
# --- Node 1: Gather & acknowledge new sources ---
gather_prompt = ChatPromptTemplate.from_template(
    """A researcher is building a brief on: {topic}

Session #{session_num}. They've provided new material:

{new_context}

Summarize what new material has been provided and what it covers.
Keep it to 2-3 sentences.

Source summary:"""
)


def gather_sources(state: ResearchState) -> dict:
    session = state.get("session_number", 0) + 1
    print(f"📚 Node: gather_sources  (Session {session})")
    chain = gather_prompt | llm | StrOutputParser()
    source_summary = chain.invoke({
        "topic": state["topic"],
        "session_num": session,
        "new_context": state["new_context"]
    })
    return {
        "session_number": session,
        "session_log": [f"Session {session}: {source_summary[:100]}..."]
    }


# --- Node 2: Extract insights from new material ---
extract_prompt = ChatPromptTemplate.from_template(
    """Extract key insights from this new research material about: {topic}

New Material:
{new_context}

Existing Brief (from prior sessions):
{existing_brief}

Extract insights that are NEW — not already covered in the existing brief.
For each insight:
- FINDING: What was discovered
- SIGNIFICANCE: Why it matters
- CONFIDENCE: high / medium / low (based on source quality)

New insights:"""
)


def extract_insights(state: ResearchState) -> dict:
    print("🔍 Node: extract_insights")
    chain = extract_prompt | llm | StrOutputParser()
    insights = chain.invoke({
        "topic": state["topic"],
        "new_context": state["new_context"],
        "existing_brief": state.get("running_brief", "(No prior brief — this is the first session)")
    })
    return {"latest_insights": insights}


# --- Node 3: Update the running brief ---
update_prompt = ChatPromptTemplate.from_template(
    """Update the running research brief on: {topic}

EXISTING BRIEF:
{existing_brief}

NEW INSIGHTS from this session:
{new_insights}

Create an UPDATED brief that:
1. Preserves all valid prior findings
2. Integrates the new insights
3. Resolves any contradictions (note the contradiction)
4. Maintains a clear, organized structure
5. Keeps a running count of sessions and sources reviewed

Updated brief:"""
)


def update_brief(state: ResearchState) -> dict:
    print("📝 Node: update_brief")
    chain = update_prompt | llm | StrOutputParser()
    updated = chain.invoke({
        "topic": state["topic"],
        "existing_brief": state.get("running_brief", "(First session — no prior brief)"),
        "new_insights": state["latest_insights"]
    })
    return {"running_brief": updated}


# --- Node 4: Identify gaps ---
gaps_prompt = ChatPromptTemplate.from_template(
    """Review the current research brief on: {topic}

Current Brief:
{brief}

Sessions completed: {session_num}

Identify:
1. KNOWLEDGE GAPS: What important aspects of the topic are not yet covered?
2. WEAK AREAS: Where is the evidence thin or contradictory?
3. SUGGESTED NEXT SOURCES: What should the researcher look at next?

Keep this actionable — suggest specific search queries or source types.

Gap analysis:"""
)


def identify_gaps(state: ResearchState) -> dict:
    print("🔎 Node: identify_gaps")
    chain = gaps_prompt | llm | StrOutputParser()
    gaps = chain.invoke({
        "topic": state["topic"],
        "brief": state["running_brief"],
        "session_num": state["session_number"]
    })
    return {"gaps": gaps}


print("All 4 nodes defined")

## Step 4 — Build Graph with MemorySaver

The `MemorySaver` checkpointer stores state after every node.
When we `invoke()` again with the **same `thread_id`**, LangGraph
resumes from the last checkpoint — so `running_brief` carries over.

In [ ]:
workflow = StateGraph(ResearchState)

workflow.add_node("gather_sources", gather_sources)
workflow.add_node("extract_insights", extract_insights)
workflow.add_node("update_brief", update_brief)
workflow.add_node("identify_gaps", identify_gaps)

workflow.set_entry_point("gather_sources")
workflow.add_edge("gather_sources", "extract_insights")
workflow.add_edge("extract_insights", "update_brief")
workflow.add_edge("update_brief", "identify_gaps")
workflow.add_edge("identify_gaps", END)

# MemorySaver persists state across invocations
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

# Reuse this config for all sessions on the same topic
config = {"configurable": {"thread_id": "research-transformers-2024"}}
print("Research workflow compiled with MemorySaver persistence")

## Step 5 — Session 1: Initial Research

We provide background context on the topic. The brief starts from scratch.

In [ ]:
session1_context = """
Topic: Transformer Architecture Evolution (2017-2024)

Source 1 - "Attention Is All You Need" (Vaswani et al., 2017):
- Introduced the Transformer model, replacing RNNs/LSTMs for sequence tasks.
- Key innovation: multi-head self-attention mechanism.
- Encoder-decoder architecture with positional encodings.
- Achieved state-of-the-art on machine translation tasks.
- Training uses teacher forcing and label smoothing.

Source 2 - BERT (Devlin et al., 2018):
- Bidirectional Encoder Representations from Transformers.
- Pre-training on Masked Language Model + Next Sentence Prediction.
- Fine-tuning paradigm: pre-train once, fine-tune for downstream tasks.
- Set new benchmarks on 11 NLP tasks including SQuAD and GLUE.

Source 3 - GPT-2 (Radford et al., 2019):
- Decoder-only transformer, 1.5B parameters.
- Showed that language models can learn tasks without explicit training.
- Zero-shot and few-shot capabilities emerged at scale.
- Raised ethical concerns about text generation misuse.
"""

result1 = app.invoke(
    {
        "topic": "Transformer Architecture Evolution (2017-2024)",
        "new_context": session1_context,
        "latest_insights": "",
        "running_brief": "",
        "gaps": "",
        "session_log": [],
        "session_number": 0,
    },
    config
)

print("\n" + "="*60)
print("📋 RESEARCH BRIEF AFTER SESSION 1")
print("="*60)
print(result1["running_brief"])
print("\n🔎 GAPS TO EXPLORE:")
print(result1["gaps"])

## Step 6 — Session 2: Add More Sources

Same `thread_id` → LangGraph loads the state from Session 1.
The `running_brief` already contains the initial research.

In [ ]:
session2_context = """
Source 4 - GPT-3 (Brown et al., 2020):
- 175B parameters, 96 attention layers.
- Demonstrated strong few-shot learning from in-context examples.
- Powers applications like code generation, summarization, translation.
- Training cost estimated at $4.6M on a cluster of V100 GPUs.

Source 5 - Vision Transformer / ViT (Dosovitskiy et al., 2020):
- Applied transformer architecture to computer vision.
- Images split into 16x16 patches, flattened and treated as tokens.
- Matched or beat CNNs on ImageNet when pre-trained on large datasets.
- Opened the door to unified architectures across modalities.

Source 6 - Flash Attention (Dao et al., 2022):
- Hardware-aware attention algorithm optimizing GPU memory access.
- Reduces attention memory from O(n²) to O(n) by tiling and recomputation.
- 2-4x speedup on long sequences without approximation.
- Enabled training on much longer context windows.
"""

result2 = app.invoke(
    {
        "topic": "Transformer Architecture Evolution (2017-2024)",
        "new_context": session2_context,
        "latest_insights": "",
        "running_brief": "",    # Will be loaded from checkpoint!
        "gaps": "",
        "session_log": [],      # Append-only — new entries are added
        "session_number": 0,
    },
    config
)

print("\n" + "="*60)
print("📋 RESEARCH BRIEF AFTER SESSION 2")
print("="*60)
print(result2["running_brief"])
print("\n🔎 GAPS REMAINING:")
print(result2["gaps"])

## Step 7 — Session 3: Final Sources & Brief

In [ ]:
session3_context = """
Source 7 - LLaMA & Open-Source LLMs (Meta, 2023):
- LLaMA showed competitive performance at smaller scales (7B-65B).
- Sparked an open-source LLM revolution: Alpaca, Vicuna, Mistral.
- Key insight: data quality > model size (Chinchilla scaling laws).

Source 8 - Mixture of Experts / MoE (Jiang et al., 2024):
- Mixtral 8x7B: 8 expert sub-networks, 2 active per token.
- Total 46.7B params but inference cost of a 12.9B model.
- Sparse activation: each token only uses a subset of experts.
- Achieves efficiency gains without sacrificing performance.

Source 9 - State Space Models / Mamba (Gu & Dao, 2023):
- Challenges the dominance of attention for long sequences.
- Linear-time alternative to quadratic attention.
- Selective state spaces: input-dependent filtering.
- Competitive with transformers on language tasks < 1B scale.
- Hybrid models (Mamba + Attention) showing best of both worlds.
"""

result3 = app.invoke(
    {
        "topic": "Transformer Architecture Evolution (2017-2024)",
        "new_context": session3_context,
        "latest_insights": "",
        "running_brief": "",
        "gaps": "",
        "session_log": [],
        "session_number": 0,
    },
    config
)

print("\n" + "="*60)
print("📋 FINAL RESEARCH BRIEF (3 sessions)")
print("="*60)
print(result3["running_brief"])
print("\n🔎 REMAINING GAPS:")
print(result3["gaps"])

## Step 8 — Inspect Session History

Because `session_log` uses `Annotated[list, operator.add]`,
all session entries are accumulated across checkpoints.

In [ ]:
print("📜 Session Log (accumulated across all sessions):")
for entry in result3.get("session_log", []):
    print(f"  • {entry}")

print(f"\n📊 Total sessions: {result3['session_number']}")

# Inspect checkpoint history
print("\n🕰️ Checkpoint History:")
for i, snapshot in enumerate(app.get_state_history(config)):
    meta = snapshot.metadata or {}
    step = meta.get("step", "?")
    node = meta.get("source", "?")
    print(f"  Step {step}: {node}")
    if i >= 15:  # Limit output
        print("  ...")
        break

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **MemorySaver** | Stores state after every node; persists across `invoke()` calls |
| **thread_id** | Same thread = same conversation; different thread = fresh start |
| **Annotated[list, add]** | Append-only state field — entries accumulate, never replaced |
| **Multi-session merge** | `update_brief` node merges new insights into existing brief |
| **Gap analysis** | Each session identifies what's still missing → guides next session |

## 🔧 Extensions

- **SQLite persistence**: Replace MemorySaver with SqliteSaver for disk persistence
- **Citation tracking**: Track which source each finding came from
- **Confidence scoring**: Weight findings by source quality and agreement
- **Export**: Generate a formatted PDF/Markdown report from final brief